# 2035 PCM Sanity Check — 30-Day Test Run

**Scenario:** GTEP Stage 3 solution → Prescient PTDF simulation  
**Period:** January 1–30, 2035 (30-day test)  
**Generators:** 278 (from `Prescient_2_2035/gen.csv`)  
**Purpose:** Validate fuel price fix before committing to 365-day run  

Key question: **Does the dispatch show correct merit order and reasonable LMPs?**

In [ ]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

plt.rcParams.update({'figure.figsize': (14, 5), 'figure.dpi': 100})

# ── Paths ────────────────────────────────────────────────────────────────────
DATA_ROOT = Path(os.path.abspath('')).parent / 'data' / 'retirement_allowed_no_extreme_half_load'
RESULTS_2035 = DATA_ROOT / 'Prescient_2_2035' / 'results'
GEN_CSV      = DATA_ROOT / 'Prescient_2_2035' / 'gen.csv'

print(f'Results dir: {RESULTS_2035}')
print(f'Results exist: {RESULTS_2035.exists()}')

# ── Helpers ───────────────────────────────────────────────────────────────────
def _prescient_output_to_df(file_name):
    """Load Prescient output CSV and combine Date/Hour/Minute into Datetime."""
    df = pd.read_csv(file_name)
    if 'Minute' in df.columns:
        df['Datetime'] = (
            pd.to_datetime(df['Date'])
            + pd.to_timedelta(df['Hour'], 'hour')
            + pd.to_timedelta(df['Minute'], 'minute')
        )
        df.drop(columns=['Date', 'Hour', 'Minute'], inplace=True)
    elif 'Hour' in df.columns:
        df['Datetime'] = (
            pd.to_datetime(df['Date'])
            + pd.to_timedelta(df['Hour'], 'hour')
        )
        df.drop(columns=['Date', 'Hour'], inplace=True)
    else:
        df['Datetime'] = pd.to_datetime(df['Date'])
        df.drop(columns=['Date'], inplace=True)
    cols = df.columns.tolist()
    cols = cols[-1:] + cols[:-1]
    return df[cols]

# ── Load data ─────────────────────────────────────────────────────────────────
daily   = pd.read_csv(RESULTS_2035 / 'daily_summary.csv', parse_dates=['Date'])
hourly  = _prescient_output_to_df(RESULTS_2035 / 'hourly_summary.csv')
bus_df  = _prescient_output_to_df(RESULTS_2035 / 'bus_detail.csv')
therm   = _prescient_output_to_df(RESULTS_2035 / 'thermal_detail.csv')
renew   = _prescient_output_to_df(RESULTS_2035 / 'renewables_detail.csv')
gen     = pd.read_csv(GEN_CSV)
overall = pd.read_csv(RESULTS_2035 / 'overall_simulation_output.csv')

print(f'Daily rows:   {len(daily):,}')
print(f'Hourly rows:  {len(hourly):,}')
print(f'Bus rows:     {len(bus_df):,}')
print(f'Thermal rows: {len(therm):,}')
print(f'Renew rows:   {len(renew):,}')
print(f'Generators:   {len(gen):,}')

## 1. Overall Metrics Dashboard

In [ ]:
# ── Overall simulation output ────────────────────────────────────────────────
o = overall.iloc[0]

# Load-weighted system LMP
bus_df['DemandLMP'] = bus_df['Demand'] * bus_df['LMP DA']
total_demand = bus_df['Demand'].sum()
lw_lmp = bus_df['DemandLMP'].sum() / total_demand

metrics = pd.DataFrame({
    'Metric': [
        'Total Demand (GWh)',
        'Total Costs ($M)',
        'Fixed Costs ($M)',
        'Generation Costs ($M)',
        'Avg Price ($/MWh) [Prescient]',
        'Load-Weighted LMP ($/MWh)',
        'Renewables Penetration (%)',
        'Load Shedding (MWh)',
        'Renewables Curtailment (MWh)',
        'Reserve Shortfall (MWh)',
        'Max Observed Demand (MW)',
        'Total On/Offs',
    ],
    'Value': [
        f"{o['Total demand'] / 1e6:.2f}",
        f"{o['Total costs'] / 1e6:.1f}",
        f"{o['Total fixed costs'] / 1e6:.1f}",
        f"{o['Total generation costs'] / 1e6:.1f}",
        f"{o['Cumulative average price']:.2f}",
        f"{lw_lmp:.2f}",
        f"{o['Overall renewables penetration rate']:.2f}",
        f"{o['Total load shedding']:.1f}",
        f"{o['Total renewables curtailment']:.1f}",
        f"{o['Total reserve shortfall']:.2f}",
        f"{o['Maximum observed demand']:.0f}",
        f"{o['Total on/offs']:.0f}",
    ],
})

print('=' * 55)
print('  2035 PCM — 30-Day Test Summary')
print('=' * 55)
for _, row in metrics.iterrows():
    print(f"  {row['Metric']:<40s} {row['Value']:>10s}")
print('=' * 55)

# Quick sanity flags
flags = []
if o['Total load shedding'] > 0:
    flags.append('⚠️ LOAD SHEDDING DETECTED')
if o['Total renewables curtailment'] > 0:
    flags.append('⚠️ RENEWABLES CURTAILMENT DETECTED')
if lw_lmp < 10 or lw_lmp > 60:
    flags.append(f'⚠️ LMP out of expected range: ${lw_lmp:.2f}/MWh')
if not flags:
    print('\n✅ All basic checks pass')
else:
    for f in flags:
        print(f'\n{f}')

## 2. Marginal Cost Validation — THE Critical Check

Verify that the fuel price fix produced correct marginal costs:  
- **NUC** ≈ \$7.38/MWh  
- **COAL** ≈ \$18.94/MWh  
- **CT (Gas)** ≈ \$22.80/MWh

In [ ]:
# ── Merge thermal dispatch with gen.csv for fuel type ────────────────────────
gen_info = gen[['GEN UID', 'Unit Type', 'Fuel', 'Fuel Price $/MMBTU',
                'HR_incr_1', 'PMax MW', 'PMin MW']].copy()
gen_info['GEN UID'] = gen_info['GEN UID'].astype(str)
gen_info['expected_mc'] = gen_info['Fuel Price $/MMBTU'] * gen_info['HR_incr_1'] * 0.001

# Only dispatching hours (Unit State == True, Dispatch > 0)
therm['Generator'] = therm['Generator'].astype(str)
dispatching = therm[therm['Dispatch'] > 0.1].copy()
dispatching = dispatching.merge(gen_info, left_on='Generator', right_on='GEN UID', how='left')

# Observed marginal cost = Unit Cost / Dispatch
dispatching['observed_mc'] = dispatching['Unit Cost'] / dispatching['Dispatch']

# Summarize by Unit Type
mc_summary = dispatching.groupby('Unit Type').agg(
    count=('observed_mc', 'size'),
    obs_mc_mean=('observed_mc', 'mean'),
    obs_mc_median=('observed_mc', 'median'),
    obs_mc_std=('observed_mc', 'std'),
    expected_mc=('expected_mc', 'mean'),
    total_gen_gwh=('Dispatch', lambda x: x.sum() / 1e3),
).round(2)

print('Marginal Cost Validation by Unit Type')
print('=' * 80)
print(mc_summary.to_string())
print()

# ── Check each fuel type ─────────────────────────────────────────────────────
expected = {'NUC': 7.38, 'COAL': 18.94, 'CT': 22.80}
for utype, exp_mc in expected.items():
    if utype in mc_summary.index:
        obs = mc_summary.loc[utype, 'obs_mc_median']
        pct_err = abs(obs - exp_mc) / exp_mc * 100
        status = '✅' if pct_err < 20 else '❌'
        print(f'{status} {utype}: observed median ${obs:.2f}/MWh vs expected ${exp_mc:.2f}/MWh ({pct_err:.1f}% diff)')
    else:
        print(f'⚠️ {utype}: no dispatching hours found')

## 3. Dispatch Mix by Fuel Type

In [ ]:
# ── Thermal generation by type ──────────────────────────────────────────────
therm_merged = therm.merge(gen_info[['GEN UID', 'Unit Type', 'PMax MW']],
                           left_on='Generator', right_on='GEN UID', how='left')

# Renewable types from gen.csv
renew_gen = gen[gen['Unit Type'].isin(['WIND', 'PV', 'HYDRO'])][['GEN UID', 'Unit Type', 'PMax MW']].copy()
renew_gen['GEN UID'] = renew_gen['GEN UID'].astype(str)
renew['Generator'] = renew['Generator'].astype(str)
renew_merged = renew.merge(renew_gen, left_on='Generator', right_on='GEN UID', how='left')

# Total generation by type (GWh)
thermal_by_type = therm_merged.groupby('Unit Type')['Dispatch'].sum() / 1e3
renew_by_type = renew_merged.groupby('Unit Type')['Output'].sum() / 1e3
gen_by_type = pd.concat([thermal_by_type, renew_by_type]).sort_values(ascending=False)
gen_by_type.name = 'Generation (GWh)'

# Installed capacity by type (MW)
cap_thermal = gen[gen['Unit Type'].isin(['NUC', 'COAL', 'CT'])].groupby('Unit Type')['PMax MW'].sum()
cap_renew = gen[gen['Unit Type'].isin(['WIND', 'PV', 'HYDRO'])].groupby('Unit Type')['PMax MW'].sum()
cap_by_type = pd.concat([cap_thermal, cap_renew])

# Capacity factor = generation / (capacity * hours)
n_hours = len(hourly)
cf = (gen_by_type * 1e3) / (cap_by_type * n_hours) * 100

summary = pd.DataFrame({
    'Generation (GWh)': gen_by_type.round(1),
    'Share (%)': (gen_by_type / gen_by_type.sum() * 100).round(1),
    'Capacity (MW)': cap_by_type.round(0),
    'Cap Factor (%)': cf.round(1),
}).sort_values('Generation (GWh)', ascending=False)

print('Generation Mix — 30-Day Test')
print('=' * 65)
print(summary.to_string())
print(f'\nTotal generation: {gen_by_type.sum():.1f} GWh')

# Merit order check
print('\n── Merit Order Check ──')
if 'NUC' in cf.index and 'COAL' in cf.index:
    nuc_cf = cf.get('NUC', 0)
    coal_cf = cf.get('COAL', 0)
    ct_cf = cf.get('CT', 0)
    if nuc_cf > coal_cf > ct_cf:
        print(f'✅ NUC ({nuc_cf:.0f}%) > COAL ({coal_cf:.0f}%) > CT ({ct_cf:.0f}%) — correct merit order')
    else:
        print(f'⚠️ Merit order: NUC={nuc_cf:.0f}% COAL={coal_cf:.0f}% CT={ct_cf:.0f}% — check dispatch')

## 4. Daily Generation Mix (Stacked Area)

In [ ]:
# ── Build daily generation by fuel type ──────────────────────────────────────
therm_merged['date'] = therm_merged['Datetime'].dt.normalize()
renew_merged['date'] = renew_merged['Datetime'].dt.normalize()

daily_thermal = therm_merged.groupby(['date', 'Unit Type'])['Dispatch'].sum().unstack(fill_value=0)
daily_renew = renew_merged.groupby(['date', 'Unit Type'])['Output'].sum().unstack(fill_value=0)
daily_gen = pd.concat([daily_thermal, daily_renew], axis=1).fillna(0)

# Reorder for nice stacking: NUC, COAL, CT, HYDRO, WIND, PV
order = [c for c in ['NUC', 'COAL', 'CT', 'HYDRO', 'WIND', 'PV'] if c in daily_gen.columns]
daily_gen = daily_gen[order] / 1e3  # Convert to GWh

colors = {'NUC': '#e41a1c', 'COAL': '#555555', 'CT': '#ff7f00',
          'HYDRO': '#377eb8', 'WIND': '#4daf4a', 'PV': '#ffff33'}
plot_colors = [colors.get(c, '#999999') for c in order]

fig, ax = plt.subplots(figsize=(14, 6))
daily_gen.plot.area(ax=ax, color=plot_colors, alpha=0.8, linewidth=0.5)
ax.set_ylabel('Daily Generation (GWh)')
ax.set_xlabel('')
ax.set_title('2035 PCM — Daily Generation Mix by Fuel Type (Jan 2035)')
ax.legend(loc='upper right', ncol=len(order))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))
plt.tight_layout()
plt.show()

## 5. LMP Analysis

In [ ]:
# ── Load-weighted system LMP by hour ─────────────────────────────────────────
hourly_lmp = (
    bus_df.groupby('Datetime')
    .apply(lambda g: (g['Demand'] * g['LMP DA']).sum() / g['Demand'].sum()
           if g['Demand'].sum() > 1.0 else g['LMP DA'].mean())
    .rename('LW_LMP')
)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# (a) Hourly LMP time series
axes[0].plot(hourly_lmp.index, hourly_lmp.values, linewidth=0.6, alpha=0.8)
axes[0].axhline(lw_lmp, color='red', ls='--', lw=1, label=f'Mean ${lw_lmp:.1f}')
axes[0].set_ylabel('LMP ($/MWh)')
axes[0].set_title('Hourly Load-Weighted LMP')
axes[0].legend()
axes[0].xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))

# (b) Daily average LMP
daily_lmp = hourly_lmp.groupby(hourly_lmp.index.normalize()).mean()
axes[1].bar(daily_lmp.index, daily_lmp.values, width=0.8, alpha=0.7)
axes[1].axhline(lw_lmp, color='red', ls='--', lw=1)
axes[1].set_ylabel('LMP ($/MWh)')
axes[1].set_title('Daily Average LMP')
axes[1].xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))

# (c) LMP histogram
axes[2].hist(hourly_lmp.values, bins=50, alpha=0.7, edgecolor='black', linewidth=0.3)
axes[2].axvline(lw_lmp, color='red', ls='--', lw=1, label=f'Mean ${lw_lmp:.1f}')
axes[2].set_xlabel('LMP ($/MWh)')
axes[2].set_ylabel('Hours')
axes[2].set_title('LMP Distribution')
axes[2].legend()

plt.tight_layout()
plt.show()

# Summary stats
neg_lmp = (hourly_lmp < 0).sum()
print(f'LMP Statistics:')
print(f'  Load-weighted mean: ${lw_lmp:.2f}/MWh')
print(f'  Median:             ${hourly_lmp.median():.2f}/MWh')
print(f'  Min:                ${hourly_lmp.min():.2f}/MWh')
print(f'  Max:                ${hourly_lmp.max():.2f}/MWh')
print(f'  Std:                ${hourly_lmp.std():.2f}')
print(f'  Negative LMP hours: {neg_lmp}')

## 6. Hourly Supply–Demand Balance

In [ ]:
# ── Hourly thermal + renewable generation vs demand ──────────────────────────
hourly_thermal = therm_merged.groupby('Datetime')['Dispatch'].sum()
hourly_renew = renew_merged.groupby('Datetime')['Output'].sum()
hourly_demand = hourly['Demand']
hourly_demand.index = hourly['Datetime']

fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

# (a) Supply vs demand
total_gen_hourly = hourly_thermal.add(hourly_renew, fill_value=0)
axes[0].plot(hourly_demand.index, hourly_demand.values, label='Demand', lw=1)
axes[0].plot(total_gen_hourly.index, total_gen_hourly.values, label='Total Gen', lw=0.8, alpha=0.7)
axes[0].set_ylabel('MW')
axes[0].set_title('System Supply vs Demand')
axes[0].legend()

# (b) Thermal vs renewable share
axes[1].fill_between(hourly_thermal.index, 0, hourly_thermal.values,
                      alpha=0.6, label='Thermal', color='#ff7f00')
axes[1].fill_between(hourly_renew.index, hourly_thermal.values,
                      hourly_thermal.values + hourly_renew.values,
                      alpha=0.6, label='Renewables', color='#4daf4a')
axes[1].plot(hourly_demand.index, hourly_demand.values, 'k-', lw=0.8, label='Demand')
axes[1].set_ylabel('MW')
axes[1].set_xlabel('')
axes[1].set_title('Thermal vs Renewable Generation')
axes[1].legend()
axes[1].xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))

plt.tight_layout()
plt.show()

# Balance check
imbalance = (total_gen_hourly - hourly_demand).abs()
print(f'Supply-demand balance:')
print(f'  Max imbalance:  {imbalance.max():.1f} MW')
print(f'  Mean imbalance: {imbalance.mean():.1f} MW')

## 7. Comparison with 2019 Baseline

Reference values from the 90-day 2019 PTDF baseline (MEMORY.md: annual load-wt LMP \$24.73/MWh).

In [ ]:
# ── Side-by-side comparison ──────────────────────────────────────────────────
# 2019 baseline values from prior analysis (90-day run)
baseline_2019 = {
    'Load-Wt LMP ($/MWh)':     24.73,
    'Renewables Pen. (%)':     None,  # Not directly comparable (different gen fleet)
    'Load Shedding (MWh)':     0.0,
    'Curtailment (MWh)':       0.0,
}

values_2035 = {
    'Load-Wt LMP ($/MWh)':     lw_lmp,
    'Renewables Pen. (%)':     o['Overall renewables penetration rate'],
    'Load Shedding (MWh)':     o['Total load shedding'],
    'Curtailment (MWh)':       o['Total renewables curtailment'],
}

comp = pd.DataFrame({
    'Metric': list(values_2035.keys()),
    '2019 Baseline (90d)': [f'{v:.2f}' if v is not None else 'N/A' for v in baseline_2019.values()],
    '2035 Test (30d Jan)': [f'{v:.2f}' for v in values_2035.values()],
})

print('Comparison: 2019 Baseline vs 2035 Test')
print('=' * 60)
print(comp.to_string(index=False))
print()
print('Note: 2019 is 90-day (Apr-Jun), 2035 is 30-day (Jan only).')
print('Direct comparison is indicative, not apples-to-apples.')

## Go / No-Go Summary

**Checklist for 365-day run approval:**

| Check | Status |
|-------|--------|
| No load shedding | |
| Marginal costs match fuel_cost3 | |
| Merit order: NUC < COAL < CT | |
| LMP in reasonable range ($15–35/MWh) | |
| Nuclear dispatches as baseload | |
| Supply-demand balance | |

**Decision:** _Fill in after reviewing results above._